In [ ]:
import numpy as np
from scipy.special import jv
import sys
sys.path.insert(0, "/Users/tmalas/Desktop/Dev/Simulations_local/Code_python")
import matplotlib.pyplot as plt
import Plot_setting

INF = np.inf

hbar = 1
e = 1
#beta = INF  # Zero-temperature case: T = 0, beta -> inf
phi_quantum = 1 / 2

# Circuit parameters
omega_a = 25.338776456203686
omega_b = 2 * omega_a

phi_a = 0.11
phi_b = 0.204

E_J = 37.12 * 2 * np.pi

# Environment
kappa_b = 5 / 10.4  # 1/ns from Leon's paper

omega_d = omega_b                  # Drive frequency
omega_p = 2 * omega_a - omega_d   # Pump frequency

epsilon_p = 0.1
g2 = jv(1, epsilon_p) * E_J * phi_a**2 * phi_b  # Leon's paper
epsilon_d = 4 * g2

g = E_J * np.sin(epsilon_p) * phi_a**2 * phi_b
epsilon_d_ratio = 2
params = (epsilon_p, E_J, epsilon_d_ratio, kappa_b)

# $\kappa_a = 0$

In [ ]:
import numpy as np
from scipy.interpolate import interp1d

# parameters
# assume g, epsilon_d, kappa_b already defined
A_0 = np.sqrt(epsilon_d / g)
B_0 = np.sqrt(2) * epsilon_d / kappa_b

# reduced vector field X = (u, B), where u = P_A^2
def F(X):
    u, B = X
    du = 2*np.sqrt(2) * g * u * B
    dB = -epsilon_d/np.sqrt(2) + g*u/np.sqrt(2) - 0.5*kappa_b*B
    return np.array([du, dB])

def tangent(string):
    T = np.zeros_like(string)
    T[1:-1] = string[2:] - string[:-2]
    T[0] = string[1] - string[0]
    T[-1] = string[-1] - string[-2]
    norms = np.linalg.norm(T, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return T / norms

def reparametrize(string):
    ds = np.linalg.norm(np.diff(string, axis=0), axis=1)
    s = np.zeros(len(string))
    s[1:] = np.cumsum(ds)

    if s[-1] == 0:
        return string.copy()

    s_uniform = np.linspace(0, s[-1], len(string))

    u_interp = interp1d(s, string[:, 0], kind='linear')
    B_interp = interp1d(s, string[:, 1], kind='linear')

    new_string = np.zeros_like(string)
    new_string[:, 0] = u_interp(s_uniform)
    new_string[:, 1] = B_interp(s_uniform)
    return new_string

def initialize_string(N):
    # adiabatic curve: B_ad(u) = sqrt(2)/kappa_b * (g*u - epsilon_d)
    u_vals = np.linspace(A_0**2, 0.0, N)
    B_vals = (np.sqrt(2)/kappa_b) * (g*u_vals - epsilon_d)
    return np.column_stack([u_vals, B_vals])

def string_step(string, dtau):
    new_string = string.copy()
    T = tangent(string)

    for i in range(1, len(string)-1):
        Fi = F(string[i])

        # normal component only
        Fi_normal = Fi - np.dot(Fi, T[i]) * T[i]

        # relaxation sign
        new_string[i] += dtau * Fi_normal

    # fix endpoints
    new_string[0]  = np.array([A_0**2, 0.0])
    new_string[-1] = np.array([0.0, -B_0])

    # redistribute images
    new_string = reparametrize(new_string)

    # fix endpoints again after interpolation
    new_string[0]  = np.array([A_0**2, 0.0])
    new_string[-1] = np.array([0.0, -B_0])

    return new_string

def residual_norm(string):
    T = tangent(string)
    vals = []
    for i in range(1, len(string)-1):
        Fi = F(string[i])
        Fi_normal = Fi - np.dot(Fi, T[i]) * T[i]
        vals.append(np.linalg.norm(Fi_normal))
    return max(vals) if vals else 0.0

def string_method(N=200, dtau=1e-3, max_iter=20000, tol=1e-8):
    string = initialize_string(N)

    for k in range(max_iter):
        old = string.copy()
        string = string_step(string, dtau)

        err = np.max(np.linalg.norm(string - old, axis=1))
        res = residual_norm(string)

        if k % 200 == 0:
            print(f"iter={k}, move={err:.3e}, res={res:.3e}")

        if err < tol and res < tol:
            print(f"converged at iter={k}, move={err:.3e}, res={res:.3e}")
            break

    return string

In [ ]:
N=200
string_init = initialize_string(N)
string_test = string_method(dtau=1e-2)

In [ ]:
plt.plot(string_test[:,0],string_test[:,1])
plt.plot(string_init[:,0],string_init[:,1])
plt.xlabel(r'$P_A$')
plt.ylabel(r'$B$')
plt.show()

# $\kappa_a\neq 0$

In [ ]:
kappa_2 = 4 * g**2 / kappa_b
kappa_a = 0.5 * kappa_2

delta = kappa_a * kappa_b / 4 / g**2
a = (A_0**2)/(2*A_0**2 - 2*delta)*(1 + (2*delta)/(A_0**2)-np.sqrt(1+4*(2*delta)/(A_0**2)))
P_A_value = np.sqrt(delta/(a*(a**2-1)))
A_value = a*P_A_value
B_value = -2*np.sqrt(2)*g/kappa_b*A_value**2-kappa_a/(2*np.sqrt(2)*g)
P_B_value = -2*np.sqrt(2)*g*A_value*P_A_value/kappa_b

In [ ]:
c=np.sqrt(2)*g

def reparametrize(string):
    # arc-length parametrization
    ds = np.linalg.norm(np.diff(string, axis=0), axis=1)
    s = np.zeros(len(string))
    s[1:] = np.cumsum(ds)

    if s[-1] == 0:
        return string.copy()

    s_uniform = np.linspace(0, s[-1], len(string))

    A_interp = interp1d(s, string[:, 0], kind='linear')
    PA_interp = interp1d(s, string[:, 1], kind='linear')
    B_interp = interp1d(s, string[:, 2], kind='linear')
    PB_interp = interp1d(s, string[:, 3], kind='linear')

    new_string = np.zeros_like(string)
    new_string[:, 0] = A_interp(s_uniform)
    new_string[:, 1] = PA_interp(s_uniform)
    new_string[:, 2] = B_interp(s_uniform)
    new_string[:, 3] = PB_interp(s_uniform)
    return new_string

def initialize_string(N):
    # better than a straight line: use adiabatic curve
    PA_vals = np.linspace(P_A_value,0.0, N)
    A_vals = np.linspace(A_value,0.0, N)
    B_vals = -np.sqrt(2)*epsilon_d/kappa_b + np.sqrt(2)*g*(A_vals**2 + PA_vals**2)/kappa_b + 4*np.sqrt(2)*g*A_vals*PA_vals/kappa_b
    PB_vals = -2*np.sqrt(2)*g*A_vals*PA_vals/kappa_b
    return np.column_stack([A_vals, PA_vals, B_vals, PB_vals])
c=np.sqrt(2)*g
def F_loss_a(X):
    A, PA, B, PB = X
    dA = c*(A*B - PA*PB) + kappa_a * (A/2+PA)
    dPA =   c * (A * PB - PA * B) - kappa_a * PA/2
    dB =  epsilon_d/np.sqrt(2) - g*(A**2 + PA**2)/np.sqrt(2) + kappa_b * (B/2 + PB)
    dPB = - c * A * PA - kappa_b * PB/2
    return np.array([dA, dPA, dB, dPB])

def tangent(string):
    T = np.zeros_like(string)
    T[1:-1] = string[2:] - string[:-2]
    T[0] = string[1] - string[0]
    T[-1] = string[-1] - string[-2]
    norms = np.linalg.norm(T, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return T / norms

def jacobian(X):
    A, PA, B, PB = X

    J = np.array([
        [c*B+kappa_a/2, -c*PB+kappa_a, c*A, -c*PA],
        [c*PB, -c*B-kappa_a/2, -c*PA, c*A],
        [-c*A, -c*PA, kappa_b/2, kappa_b],
        [-c*PA, -c*A, 0, -kappa_b/2]             
    ])

    return J

def semi_implicit_normal_update(xi, Ti, F, dtau, reg=0.0):
    """
    One semi-implicit update for a single string point xi with unit
    tangent Ti. Solves
 
        (I - dtau * J_perp) dx = dtau * F_perp(xi)
 
    instead of the explicit dx = dtau * F_perp(xi), where J_perp is F's
    local Jacobian at xi restricted to the subspace normal to Ti.
    Returns dx (which lies purely in that normal subspace, up to
    floating-point error).
 
    reg: optional small Tikhonov term added to the solve, as a safety
    valve if (I - dtau*J_perp) is ever near-singular.
    """
    xi = np.asarray(xi, dtype=float)
    n = xi.size
    I = np.eye(n)
 
    # Projector onto the subspace normal to the local tangent.
    P_perp = I - np.outer(Ti, Ti)
 
    Fi = np.asarray(F(xi), dtype=float)
    Fi_normal = P_perp @ Fi
 
    # Local Jacobian of F at xi, restricted to the normal subspace.
    Ji = jacobian(xi)
    J_perp = P_perp @ Ji @ P_perp
 
    A_mat = I - dtau * J_perp
    if reg > 0.0:
        A_mat = A_mat + reg * I
    rhs = dtau * Fi_normal
 
    dx = np.linalg.solve(A_mat, rhs)
 
    # dx should already lie purely in the normal subspace: because
    # J_perp @ Ti = 0 exactly, the tangential component of the solve
    # vanishes analytically. Reproject to strip floating-point leakage.
    return P_perp @ dx
 
 
def string_step(string, F, dtau, init_point, end_point, reg=0.0):
    """
    Semi-implicit string-method step.
 
    Same interface as the original explicit version, but each interior
    point is advanced with an implicit solve against the local transverse
    Jacobian instead of a plain dtau*F_perp kick. This is the fix
    recommended in 05_numerics_and_open_questions.md: it removes the
    exponential blow-up seen with explicit stepping, because the update
    multiplies the transverse error by 1/(1 - dtau*lambda) instead of
    (1 + dtau*lambda) for the local transverse eigenvalue lambda — bounded
    (and in fact damping, for large enough dtau) regardless of the sign of
    lambda, unlike the explicit multiplier which is unbounded for either
    sign as dtau or lambda grow.
    """
    new_string = string.copy()
    T = tangent(string)
 
    for i in range(1, len(string) - 1):
        dx = semi_implicit_normal_update(
            string[i], T[i], F, dtau, reg=reg
        )
        new_string[i] = string[i] + dx
 
    # fix endpoints
    new_string[0] = init_point
    new_string[-1] = end_point
 
    # redistribute images
    new_string = reparametrize(new_string)
 
    # fix endpoints again after interpolation
    new_string[0] = init_point
    new_string[-1] = end_point
 
    return new_string

def string_method(initialize_string, F, N=200, dtau=1e-3, max_iter=20000, tol=1e-8):
    string = initialize_string(N)
    init_point = string[0]
    end_point = string[-1]
    err_list = []
    for k in range(max_iter):
        old = string.copy()
        string = string_step(string, F, dtau, init_point, end_point)

        err = np.max(np.linalg.norm(string - old, axis=1))
        err_list.append(err)
        if k % 200 == 0:
            print(f"iter={k}, err={err:.3e}")

        if err < tol:
            print(f"converged at iter={k}, err={err:.3e}")
            break
    return string, err_list

In [ ]:
N=100
string_init = initialize_string(N)
string_test_loss, err_list = string_method(initialize_string, F_loss_a, dtau=1e-2, max_iter = 3_000)

In [ ]:
fig, axes = plt.subplots(1,2)
ax_1, ax_2 = axes

ax_1.plot(string_test_loss[:,1],string_test_loss[:,2])
ax_1.plot(string_init[:,1],string_init[:,2])
ax_1.set_xlabel(r'$P_A$')
ax_1.set_ylabel(r'$B$')

ax_2.plot(string_test_loss[:,0],string_test_loss[:,3])
ax_2.plot(string_init[:,0],string_init[:,3])
ax_2.set_xlabel(r'$A$')
ax_2.set_ylabel(r'$P_B$')

plt.tight_layout()
plt.show()